In [1]:
!nvidia-smi

Fri May 22 20:10:27 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   29C    P0             45W /  400W |       0MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [1]:
import os

# GPU and environment setup
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# Load secrets
from google.colab import userdata
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
os.environ["WANDB_API_KEY"] = userdata.get("WANDB_API_KEY")
os.environ["WANDB_PROJECT"] = "pydoc-llama"

# Clone repo
import subprocess, sys
if not os.path.isdir("/content/repo"):
    subprocess.run(
        ["git", "clone", "https://github.com/arinkc/llm-finetuning-project.git", "/content/repo"],
        check=True,
    )
else:
    subprocess.run(["git", "-C", "/content/repo", "pull"], check=True)
sys.path.insert(0, "/content/repo")

# Authenticate HF
from huggingface_hub import login
login(token=os.environ["HF_TOKEN"])

import warnings, torch
warnings.filterwarnings("ignore", category=SyntaxWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

print(f"✅ Session ready")
print(f"   GPU: {torch.cuda.get_device_name(0)}")
print(f"   VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


✅ Session ready
   GPU: NVIDIA A100-SXM4-40GB
   VRAM: 42.4 GB


In [2]:
!pip install -q --upgrade \
    "transformers>=4.45.0,<4.50.0" \
    "datasets>=3.0.0" \
    "peft>=0.13.0,<0.15.0" \
    "trl>=0.11.0,<0.13.0" \
    "bitsandbytes" \
    "accelerate>=1.0.0" \
    "wandb>=0.18.0" \
    "sentencepiece" "protobuf"

print("✅ Libraries installed — restart runtime now")

✅ Libraries installed — restart runtime now


In [2]:
!cd /content/repo && git pull
import importlib, src.train
importlib.reload(src.train)
from src.train import TrainingConfig, run_training

cfg = TrainingConfig(smoke_test=True)
print(f"GPU: {torch.cuda.get_device_name(0)}")
trainer = run_training(cfg)

Already up to date.
GPU: NVIDIA A100-SXM4-40GB
Loading tokenizer: meta-llama/Llama-3.1-8B-Instruct
Loading model: meta-llama/Llama-3.1-8B-Instruct (4-bit)


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

✅ LoRA configured: 41,943,040 trainable / 4,582,543,360 total (0.9153%)
Loading dataset: Arinkc/pydoc-llama-codesearchnet-curated
⚠️  Smoke test mode — using 100 train / 50 val examples
Pre-tokenizing dataset...


Tokenizing train (num_proc=4):   0%|          | 0/100 [00:00<?, ? examples/s]

Tokenizing validation (num_proc=4):   0%|          | 0/50 [00:00<?, ? examples/s]

   Train: 100
   Validation: 50
🚀 Starting training...


wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: kcarin123 (kcarin123-salisbury-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Step,Training Loss,Validation Loss


✅ Final adapter saved locally to /tmp/checkpoints/final


In [2]:
import importlib, src.train
importlib.reload(src.train)
from src.train import TrainingConfig, run_training

cfg = TrainingConfig(smoke_test=False)
print(f"Run: {cfg.wandb_run_name}")
print(f"Dataset: {cfg.dataset_id} (full — 22,473 examples)")
print(f"Epochs: {cfg.num_train_epochs}")
print(f"Batch: {cfg.per_device_train_batch_size} × grad_accum {cfg.gradient_accumulation_steps} = effective {cfg.per_device_train_batch_size * cfg.gradient_accumulation_steps}")
trainer = run_training(cfg)

Run: full-run-r16-lr0.0002
Dataset: Arinkc/pydoc-llama-codesearchnet-curated (full — 22,473 examples)
Epochs: 3
Batch: 2 × grad_accum 8 = effective 16
Loading tokenizer: meta-llama/Llama-3.1-8B-Instruct
Loading model: meta-llama/Llama-3.1-8B-Instruct (4-bit)


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

✅ LoRA configured: 41,943,040 trainable / 4,582,543,360 total (0.9153%)
Loading dataset: Arinkc/pydoc-llama-codesearchnet-curated
Pre-tokenizing dataset...


Tokenizing train (num_proc=4):   0%|          | 0/22473 [00:00<?, ? examples/s]

Tokenizing validation (num_proc=4):   0%|          | 0/1248 [00:00<?, ? examples/s]

...

README.md: 0.00B [00:00, ?B/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...mpo1k9mxvv/tokenizer.json:  89%|########9 | 15.3MB / 17.2MB            

✅ Pushed: https://huggingface.co/Arinkc/pydoc-llama-r16-full
